# Análisis: ¿es significativo el realce de recombinación por permanencia g(t)?

Compara varias **condiciones** (p. ej. G = 1.5 vs G = 1 vs β escalado), cada una con **varias semillas**, todas desde el mismo estado inicial y el mismo protocolo de pulsos.

Para cada simulación se usa:
- `sim_metadata_*.json` → G, τ_c, β, dt (de `ctes_dict` / `params_dict`).
- `Data_pp_reset_*.npz` → `datos_sim` (fila k = paso k; columnas `t, V, I_total, R_total, …`).
- `reset/Estado_pp_reset_sim_*_paso_*.npz` → `actual_state`, `oxygen_state`, `reloj_iones`, `probabilidad_recombinacion`.

**Recombinaciones:** se deducen de los snapshots como celdas cuya vacante pasa de 1 a 0 entre dos snapshots consecutivos (resolución = `pasos_guardar_estado`, 10 pasos). Se asume que en PP_reset solo la recombinación elimina vacantes.

Las métricas por simulación se guardan en una caché (`CACHE_DIR`). Si se vuelve a ejecutar, solo se procesan las simulaciones nuevas.

## 1. Configuración

In [ ]:
from pathlib import Path

# Cada condición -> lista de patrones glob de carpetas simulation_* (rutas absolutas o relativas a RAIZ)
RAIZ = Path(".")
CONDICIONES = {
    "A: G=1.5, τc=4e-4": ["Results_pulsos_reset_t_1-6_permanencia_estudio/simulation_*"],
    # "B: G=1 (control)":   ["Results_permanencia_G1/simulation_*"],
    # "C: G=1, β·1.21":     ["Results_permanencia_beta/simulation_*"],
    # "D: G=1.5, τc→0":     ["Results_permanencia_tauc_bajo/simulation_*"],
}
REFERENCIA = "A: G=1.5, τc=4e-4"   # condición frente a la que se calculan cocientes (normalmente el control G=1)

CACHE_DIR = Path("cache_analisis_permanencia")
FORZAR_RECALCULO = False
N_BOOT = 5000          # remuestreos bootstrap
SEMILLA_BOOT = 0

## 2. Imports y utilidades

In [ ]:
import json, re, hashlib, glob
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

CACHE_DIR.mkdir(parents=True, exist_ok=True)

COL_V, COL_R = 1, 3   # columnas de datos_sim


def g_realce(t, G, tau_c):
    """g(t) = 1 + (G-1)·t/(t+τc)  (mismo modelo que Recombination.Prob_Recombination)."""
    t = np.asarray(t, dtype=float)
    return 1.0 + (G - 1.0) * t / (t + tau_c)


def carpetas_condicion(patrones):
    # RAIZ / p devuelve p tal cual si p es absoluta
    out = [Path(c) for p in patrones for c in sorted(glob.glob(str(RAIZ / p)))]
    return [c for c in out if c.is_dir()]


def leer_metadata(carpeta):
    f = next(carpeta.glob("sim_metadata_*.json"))
    m = json.loads(f.read_text())
    ctes, params = m["ctes_dict"], m["params_dict"]
    return {
        "G": float(ctes.get("realce_max_recombinacion", 1.0)),
        "tau_c": float(ctes.get("tau_c_recombinacion", np.inf)),
        "beta": float(ctes["recom_enchancement_factor"]),
        "dt": float(params["paso_temporal"]),
        "seed": params.get("seed"),
        "cada": int(ctes.get("pasos_guardar_estado", 10)),
    }


def segmentos_pulso(V):
    """Tramos contiguos con V != 0 -> lista de (inicio, fin_exclusivo)."""
    on = (V != 0).astype(np.int8)
    d = np.diff(np.r_[0, on, 0])
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))

## 3. Procesado de cada simulación (con caché)

In [ ]:
def procesar_simulacion(carpeta):
    meta = leer_metadata(carpeta)
    datos = np.load(next(carpeta.glob("Data_pp_reset_*.npz")))["datos_sim"]
    V, R = datos[:, COL_V], datos[:, COL_R]
    n_filas = len(V)
    segs = segmentos_pulso(V)

    # Posición dentro del pulso para cada paso (-1 si V = 0)
    pos = np.full(n_filas, -1, dtype=np.int64)
    for a, b in segs:
        pos[a:b] = np.arange(b - a)

    snaps = sorted(((int(re.search(r"paso_(\d+)", f.name).group(1)), f)
                    for f in (carpeta / "reset").glob("Estado_pp_reset_*.npz")), key=lambda x: x[0])
    pasos = np.array([k for k, _ in snaps])

    n_vac = np.zeros(len(snaps), dtype=np.int64)
    n_ox = np.zeros(len(snaps), dtype=np.int64)
    n_ov = np.zeros(len(snaps), dtype=np.int64)       # iones sobre vacante
    n_rec = np.zeros(len(snaps), dtype=np.int64)      # vacantes 1->0 desde el snapshot anterior
    sumP = np.zeros(len(snaps))                       # Σ P (con g) en iones sobre vacante
    sumP_g1 = np.zeros(len(snaps))                    # Σ P/g  (misma configuración, g = 1)
    hist_reloj = np.zeros(1, dtype=np.int64)          # bincount del reloj (ion sobre vacante, V != 0)

    vac_prev = None
    for i, (k, f) in enumerate(snaps):
        d = np.load(f)
        vac, ox = d["actual_state"] == 1, d["oxygen_state"] == 1
        ov = vac & ox
        n_vac[i], n_ox[i], n_ov[i] = vac.sum(), ox.sum(), ov.sum()
        if vac_prev is not None:
            n_rec[i] = np.count_nonzero(vac_prev & ~vac)
        vac_prev = vac
        if ov.any():
            r = d["reloj_iones"][ov].astype(np.int64)
            P = np.minimum(d["probabilidad_recombinacion"][ov], 1.0)
            g = g_realce(r * meta["dt"], meta["G"], meta["tau_c"])
            sumP[i], sumP_g1[i] = P.sum(), (P / g).sum()
            if k < n_filas and V[k] != 0:
                bc = np.bincount(r)
                if len(bc) > len(hist_reloj):
                    hist_reloj = np.pad(hist_reloj, (0, len(bc) - len(hist_reloj)))
                hist_reloj[:len(bc)] += bc

    # Observables por pulso: R y vacantes al final de cada pulso (último paso con V != 0)
    fin = np.array([b - 1 for a, b in segs])
    R_pulso = R[fin]
    idx_snap = np.searchsorted(pasos, fin, side="right") - 1
    vac_pulso = n_vac[np.clip(idx_snap, 0, None)]

    return dict(
        pasos=pasos, n_vac=n_vac, n_ox=n_ox, n_ov=n_ov, n_rec=n_rec,
        pos_snap=pos[np.clip(pasos, 0, n_filas - 1)], V_snap=V[np.clip(pasos, 0, n_filas - 1)],
        sumP=sumP, sumP_g1=sumP_g1, hist_reloj=hist_reloj,
        R_pulso=R_pulso, vac_pulso=vac_pulso, R_ini=R[0], vac_ini=n_vac[0],
        long_pulso=np.array([b - a for a, b in segs]),
        **{k: np.array(v if v is not None else np.nan, dtype=float) for k, v in meta.items()},
    )


def cargar(carpeta):
    clave = hashlib.md5(str(carpeta.resolve()).encode()).hexdigest()[:12]
    f = CACHE_DIR / f"{carpeta.parent.name}__{carpeta.name}__{clave}.npz"
    if f.exists() and not FORZAR_RECALCULO:
        return dict(np.load(f))
    print(f"  procesando {carpeta} ...")
    res = procesar_simulacion(carpeta)
    np.savez_compressed(f, **res)
    return res


SIMS = {}
for nombre, patrones in CONDICIONES.items():
    carpetas = carpetas_condicion(patrones)
    print(f"{nombre}: {len(carpetas)} simulaciones")
    SIMS[nombre] = [cargar(c) for c in carpetas]

## 4. Resumen por condición
Parámetros leídos de los metadatos y métricas globales. `rec_esp_g/g1` es el cociente ΣP / Σ(P/g) sobre la misma trayectoria: el efecto que *debería* tener g si no cambiase la dinámica.

In [ ]:
def resumen(sims):
    tot = np.array([s["n_rec"].sum() for s in sims])
    return dict(
        n=len(sims), G=float(sims[0]["G"]), tau_c=float(sims[0]["tau_c"]), beta=float(sims[0]["beta"]),
        rec_media=tot.mean(), rec_std=tot.std(ddof=1) if len(tot) > 1 else np.nan,
        rec_esp_cociente=np.sum([s["sumP"].sum() for s in sims]) / np.sum([s["sumP_g1"].sum() for s in sims]),
        dR_media=np.mean([s["R_pulso"][-1] - s["R_ini"] for s in sims]),
    )

TOT = {n: np.array([s["n_rec"].sum() for s in sims]) for n, sims in SIMS.items()}
print(f"{'condición':28s} {'n':>3s} {'G':>5s} {'τc':>8s} {'β':>7s} {'rec media':>10s} {'σ':>6s} {'esp g/g1':>9s} {'ΔR media':>9s}")
for n, sims in SIMS.items():
    r = resumen(sims)
    print(f"{n:28s} {r['n']:3d} {r['G']:5.2f} {r['tau_c']:8.1e} {r['beta']:7.0f} {r['rec_media']:10.2f} "
          f"{r['rec_std']:6.2f} {r['rec_esp_cociente']:9.3f} {r['dR_media']:9.2f}")

## Plot 1: distribución del reloj y de g(t) en iones sobre vacante (V ≠ 0)
Muestra hasta dónde llega realmente el reloj. Si los iones saltan cada ~a/(v·dt) pasos, el reloj queda acotado por ese valor y g por g(t_salto).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for n, sims in SIMS.items():
    L = max(len(s["hist_reloj"]) for s in sims)
    h = sum(np.pad(s["hist_reloj"], (0, L - len(s["hist_reloj"]))) for s in sims).astype(float)
    if h.sum() == 0:
        continue
    h /= h.sum()
    r = np.arange(L)
    G, tc, dt = float(sims[0]["G"]), float(sims[0]["tau_c"]), float(sims[0]["dt"])
    # Agrupado en bloques de `cada` pasos: el muestreo cada 10 pasos con relojes sincronizados
    # produce picos espurios si se pinta paso a paso. El bin del reloj = 1 queda aparte.
    cada = int(sims[0]["cada"])
    bordes = np.r_[0, 1, 2, np.arange(cada, L + cada, cada)]
    b = np.r_[bordes[bordes < L], L]
    hb = np.add.reduceat(h, b[:-1]) / np.diff(b)   # densidad: fracción por paso de reloj
    axs[0].stairs(hb, b, label=n)
    g = g_realce(r * dt, G, tc)
    orden = np.argsort(g)
    axs[1].plot(g[orden], np.cumsum(h[orden]), label=n)
axs[0].set(xlabel="reloj del ion [pasos]", ylabel="fracción por paso", yscale="log", title="Reloj (ion sobre vacante, V≠0)")
axs[1].set(xlabel="g(t)", ylabel="ECDF", title="Realce alcanzado")
for a in axs: a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Plot 2: evolución pulso a pulso (media ± IQR entre semillas)
Vacantes y R al final de cada pulso. Es el observable principal.

In [ ]:
def banda(ax, M, label):
    M = np.asarray(M, dtype=float)
    x = np.arange(1, M.shape[1] + 1)
    ax.plot(x, np.nanmean(M, 0), label=label)
    if M.shape[0] > 1:
        ax.fill_between(x, *np.nanpercentile(M, [25, 75], 0), alpha=0.25)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for n, sims in SIMS.items():
    m = min(len(s["vac_pulso"]) for s in sims)
    banda(axs[0], [s["vac_pulso"][:m] for s in sims], n)
    banda(axs[1], [s["R_pulso"][:m] for s in sims], n)
axs[0].set(xlabel="pulso", ylabel="vacantes", title="Vacantes al final de cada pulso")
axs[1].set(xlabel="pulso", ylabel="R [Ω]", title="R al final de cada pulso")
for a in axs: a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Plot 3: recombinaciones totales por condición y significación
Boxplot + puntos (una simulación = un punto). Cociente de medias frente a `REFERENCIA` con IC 95 % bootstrap y test de Mann-Whitney.

In [ ]:
rng = np.random.default_rng(SEMILLA_BOOT)

def boot_cociente(a, b):
    if len(a) < 2 or len(b) < 2 or b.mean() == 0:
        return np.nan, (np.nan, np.nan)
    ra = rng.choice(a, (N_BOOT, len(a))).mean(1)
    rb = rng.choice(b, (N_BOOT, len(b))).mean(1)
    q = ra / np.where(rb == 0, np.nan, rb)
    return a.mean() / b.mean(), tuple(np.nanpercentile(q, [2.5, 97.5]))

nombres = list(TOT)
fig, ax = plt.subplots(figsize=(max(5, 2.2 * len(nombres)), 4))
ax.boxplot([TOT[n] for n in nombres], showfliers=False)
for i, n in enumerate(nombres, 1):
    ax.plot(i + rng.uniform(-0.12, 0.12, len(TOT[n])), TOT[n], "o", alpha=0.6)
ax.set_xticks(range(1, len(nombres) + 1), nombres, rotation=15, fontsize=8)
ax.set_ylabel("recombinaciones totales por simulación")
plt.tight_layout(); plt.show()

ref = TOT[REFERENCIA]
print(f"Referencia: {REFERENCIA}  (media {ref.mean():.2f}, n = {len(ref)})")
for n in nombres:
    if n == REFERENCIA:
        continue
    c, ic = boot_cociente(TOT[n], ref)
    p = stats.mannwhitneyu(TOT[n], ref).pvalue if len(TOT[n]) > 1 and len(ref) > 1 else np.nan
    print(f"  {n:28s} cociente = {c:.3f}  IC95 = [{ic[0]:.3f}, {ic[1]:.3f}]  Mann-Whitney p = {p:.3g}")

## Plot 4: tasa de recombinación según la posición dentro del pulso
Recombinaciones por intervalo de snapshot, sumadas sobre todos los pulsos y semillas y normalizadas por número de simulaciones. Abajo, cociente frente a `REFERENCIA` con la curva g(posición·dt) como referencia. Es solo cota superior: el reloj de cada ion se reinicia al saltar.

In [ ]:
ANCHO_BIN = 50   # pasos

def perfil(sims):
    L = int(max(s["long_pulso"].max() for s in sims))
    bins = np.arange(0, L + ANCHO_BIN, ANCHO_BIN)
    h = np.zeros(len(bins) - 1)
    for s in sims:
        m = s["pos_snap"] >= 0
        h += np.histogram(s["pos_snap"][m], bins=bins, weights=s["n_rec"][m])[0]
    return bins, h / len(sims)

fig, axs = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
P = {n: perfil(sims) for n, sims in SIMS.items()}
for n, (bins, h) in P.items():
    axs[0].step(bins[:-1], h, where="post", label=n)
axs[0].set(ylabel=f"recomb. / sim / {ANCHO_BIN} pasos", title="Recombinaciones dentro del pulso")
axs[0].legend(fontsize=8)

bins_r, h_r = P[REFERENCIA]
for n, (bins, h) in P.items():
    if n == REFERENCIA:
        continue
    k = min(len(h), len(h_r))
    with np.errstate(divide="ignore", invalid="ignore"):
        axs[1].step(bins[:k], h[:k] / h_r[:k], where="post", label=f"{n} / ref")
    s0 = SIMS[n][0]; sr = SIMS[REFERENCIA][0]
    x = bins[:k] + ANCHO_BIN / 2
    gx = g_realce(x * float(s0["dt"]), float(s0["G"]), float(s0["tau_c"])) / \
         g_realce(x * float(sr["dt"]), float(sr["G"]), float(sr["tau_c"]))
    axs[1].plot(x, gx * float(s0["beta"]) / float(sr["beta"]), "k--", lw=1)
axs[1].axhline(1, color="gray", lw=0.8)
axs[1].set(xlabel="posición dentro del pulso [pasos]", ylabel="cociente", title="Cociente frente a la referencia (-- : g·β esperado)")
axs[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Plot 5: efecto relativo por condición
Cociente de recombinaciones medias frente a `REFERENCIA` (IC 95 % bootstrap), junto al cociente esperado ΣP/Σ(P/g) calculado sobre la misma trayectoria. Con varias G o τ_c da la curva dosis-respuesta.

In [ ]:
etiquetas, c, lo, hi, esp = [], [], [], [], []
for n, sims in SIMS.items():
    cc, ic = boot_cociente(TOT[n], ref) if n != REFERENCIA else (1.0, (1.0, 1.0))
    etiquetas.append(f"{n}\nG={float(sims[0]['G']):g}, τc={float(sims[0]['tau_c']):.0e}, β={float(sims[0]['beta']):.0f}")
    c.append(cc); lo.append(cc - ic[0]); hi.append(ic[1] - cc)
    esp.append(resumen(sims)["rec_esp_cociente"])

x = np.arange(len(etiquetas))
fig, ax = plt.subplots(figsize=(max(5, 2.4 * len(x)), 4))
ax.errorbar(x, c, yerr=[lo, hi], fmt="o", capsize=4, label="observado (vs referencia)")
ax.plot(x, esp, "s", mfc="none", label="esperado ΣP/Σ(P/g) (solo g)")
ax.axhline(1, color="gray", lw=0.8)
ax.set_xticks(x, etiquetas, fontsize=8)
ax.set_ylabel("cociente de recombinaciones")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()